## Clasificar todos los genes por su fitness


In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
data_dir = "./Fitness_data"

files = glob.glob(os.path.join(data_dir, "*.tsv"))

one_rep = []
two_rep = []

for f in files:
  #  print(os.path.basename(f))
    cols = pd.read_csv(f, sep="\t", nrows=0).columns.tolist()

    glucose_cols = [c for c in cols if c.endswith("D-Glucose (C)")]

    if len(glucose_cols) == 1:
        one_rep.append(f)
    elif len(glucose_cols) == 2:
        two_rep.append(f)
    else:
        name = os.path.basename(f)
        print(f"{name}: {len(glucose_cols)} columnas de glucosa")

print("\nFiles with a rep:", len(one_rep))
print("\nFiles with two rep:", len(two_rep))

fit_Pseudomonas_simiae_WCS417.tsv: 3 columnas de glucosa
fit_Bacteroides_thetaiotaomicron.tsv: 8 columnas de glucosa
fit_Caulobacter_crescentus_NA1000.tsv: 3 columnas de glucosa
fit_Echinicola_vietnamensis.tsv: 3 columnas de glucosa
fit_Pseudomonas_putida_KT2440.tsv: 5 columnas de glucosa

Files with a rep: 1

Files with two rep: 3


In [3]:
# Crear nuevo directorio en el path actual
new_dir = "./TopGO2/orgs_with_categories"
os.makedirs(new_dir, exist_ok=True)

# Lista de archivos a procesar
files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".tsv")]

for f in files:
    # Leer encabezado
    cols = pd.read_csv(f, sep="\t", nrows=0).columns.tolist()
    
    # Filtrar columnas que terminen exactamente en 'D-Glucose (C)'
    glucose_cols = [c for c in cols if c.endswith("D-Glucose (C)")]
    
    if len(glucose_cols) == 1:        
        # Leer archivo completo
        data = pd.read_csv(f, sep="\t")
        data_sel = data[['sysName', glucose_cols[0]]].copy()  #locus_id
        data_sel.rename(columns={glucose_cols[0]: "Glucose"}, inplace=True)
        
        # Crear categorías
        conditions = [
            data_sel['Glucose'] <= -2,
            (data_sel['Glucose'] > -2) & (data_sel['Glucose'] < -0.5),
            (data_sel['Glucose'] >= -0.5) & (data_sel['Glucose'] <= 0.5),
            data_sel['Glucose'] > 0.5
        ]
        category = ['E', 'I', 'N', 'F']
        data_sel['Glucose_cat'] = np.select(conditions, category, default=np.nan)
        
        # Guardar archivo
        out_file = os.path.join(new_dir, "Categories_" + os.path.basename(f))
        data_sel.to_csv(out_file, sep="\t", index=False)
        print(f"File saved: {out_file}")
        
    elif len(glucose_cols) >= 2:
        # Leer archivo completo
        data = pd.read_csv(f, sep="\t")

        # Seleccionar locusId y las dos columnas de Glucose
        data_sel = data[['sysName']].copy()
        data_sel["Glucose"] = data[glucose_cols].mean(axis=1)

        # Crear categorías
        conditions = [
            data_sel['Glucose'] <= -2,
            (data_sel['Glucose'] > -2) & (data_sel['Glucose'] < -0.5),
            (data_sel['Glucose'] >= -0.5) & (data_sel['Glucose'] <= 0.5),
            data_sel['Glucose'] > 0.5
        ]
        category = ['E', 'I', 'N', 'F']
        data_sel['Glucose_cat'] = np.select(conditions, category, default=np.nan)

        # Guardar archivo
        out_file = os.path.join(new_dir, "Categories_" + os.path.basename(f))
        data_sel.to_csv(out_file, sep="\t", index=False)
     #   print(f"File saved: {out_file}")
      

/tmp/ipykernel_2641131/3619603012.py:38: DtypeWarning:

Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.



File saved: ./TopGO2/orgs_with_categories/Categories_fit_Sinorhizobium_meliloti_1021.tsv


In [4]:
import pandas as pd

# Función para filtrar genes
def filtrar_genes(celda):
    if pd.isna(celda):
        return ""
    genes = celda.split()  # separa por espacio
    genes_filtrados = [g for g in genes if g.startswith("PS417_")]
    return " ".join(genes_filtrados)  # los vuelve a unir en un string


uniprot = pd.read_csv('./TopGO_varios_orgs/uniprot_gos/uniprotkb_Bacteroides_thetaiotaomicron.tsv', sep='\t')
# Aplicar a la columna
uniprot["Gene Names"] = uniprot["Gene Names"].apply(filtrar_genes)
#uniprot.to_csv('uniprotkb_Pseudomonas_simiae_WCS417.tsv', sep='\t', index=False)
uniprot

,Gene Names,Gene Ontology IDs
0,,NaN
1,,NaN
2,,NaN
3,,NaN
4,,GO:0003735+ADs- GO:0006412+ADs- GO:0019843+ADs...
...,...,...
4754,,GO:0000150+ADs- GO:0003677+ADs- GO:0006310
4755,,GO:0005524
4756,,NaN
4757,,NaN


In [6]:
# elige la primera columna donde la mayoría de filas se parecen a LETRAS + (guion opcional) + números
cands = [c for c in uniprot.columns
         if uniprot[c].str.match(r'^[BT]+[_\uFF3F]?\d+$', na=False).mean() > 0.5]
id_col = cands[0] if cands else uniprot.columns[0]
print("Columna de IDs detectada:", id_col)

# Limpia: quita espacios, y elimina guiones bajos (ASCII y full-width)
uniprot[id_col] = (uniprot[id_col]
                   .str.strip()
                   .str.replace(r'[_\uFF3F]', '', regex=True))  # BT_0001 -> BT0001

# Guarda el TSV
uniprot.to_csv("universo_singuion.tsv", sep="\t", index=False)

Columna de IDs detectada: Gene Names
